# Check Connection to Database

DB Name: DESKTOP-EE25GV9\SQLEXPRESS

In [ ]:
# Function

# ========================================
# ========== For Local Computer ==========
def db_connection(server_name, database_name):
    import pandas as pd
    from sqlalchemy import create_engine

    #server_name = r"DESKTOP-EE25GV9\SQLEXPRESS" # <----- Change This
    #database_name = "stockPredictionApp" # <----- Change This

    # Do not change 'server_name' or 'database_name'
    connection_string = (
        f"mssql+pyodbc://@{server_name}/{database_name}"
        "?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )

    engine = create_engine(connection_string) # Connection to Database

    print("Connection successful!")
    
    # Test Query - Find the different tickers
    query = "SELECT * FROM Stocks"

    stocks_df = pd.read_sql(query, engine)

    print("\n",stocks_df)
    return(engine)


# ======================================
# ===== For Azure Database (Cloud) =====

# Not used to protect azure security
def azure_connection():
    
    from sqlalchemy import create_engine
    from urllib.parse import quote_plus

    server = "serverName"
    database = "DLA_StockPrediction"
    username = "username"
    password = "password"

    params = quote_plus(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"UID={username};"
        f"PWD={password};"
    )

    engine = create_engine(
        f"mssql+pyodbc:///?odbc_connect={params}"
    )

    return(engine)

# Call (Azure)
#azure_connection()

# Call
#db_connection(server_name = r"DESKTOP-EE25GV9\SQLEXPRESS", database_name = "stockPredictionApp")

In [2]:
import yfinance as yf

df = yf.download("BAC", period="1mo")
print(df.tail())


[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open    Volume
Ticker            BAC        BAC        BAC        BAC       BAC
2026-07-01  58.360001  58.480000  56.849998  57.130001  35288400
2026-07-02  58.730000  59.000000  57.939999  58.939999  29329700
2026-07-06  59.900002  59.939999  58.919998  59.139999  34645400
2026-07-07  59.860001  60.830002  59.790001  60.290001  29290400
2026-07-08  59.235001  59.564999  59.009998  59.369999   4185138


# Download Stock Data

In [3]:
# Function
def download_stockData(ticker="BAC", start_date="2020-01-01"):
    import yfinance as yf

    stock_df = yf.download(ticker, start=start_date)

    print(ticker, "Downloaded!","\n",stock_df.head())
    
    return(stock_df)

# Call
#amz = download_stockData(ticker = "AMZN") # Demo with Amazon Ticker

# Clean the Data

In [4]:
# Function
def clean_data(df, stock_id = 1):
    
    import pandas as pd
    from sqlalchemy import create_engine
    import yfinance as yf

    df.reset_index(inplace=True)

    df["StockID"] = stock_id
    #bac["PriceID"] = bac["level_0"]

    df = df[[
        "StockID",
        "index",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]]

    # Match Table in SQL
    df.columns = [
        "StockID",
        "PriceDate",
        "OpenPrice",
        "HighPrice",
        "LowPrice",
        "ClosePrice",
        "Volume"
    ]

    print(df.head())
    
    return(df)
    
# Call
#clean_data(amz)

# Insert into SQL Server

In [5]:
# Function
def insert_sql(df, engine, stock_id):
    import pandas as pd
    from sqlalchemy import create_engine
    import yfinance as yf
    
    if stock_id == 1:
        df.to_sql(
        "Prices",
        engine,
        if_exists="replace",
        index=False
    )
    else:
        df.to_sql(
        "Prices",
        engine,
        if_exists="append",
        index=False
    )
        
    

    print("Prices inserted successfully!")
    
    # Sample Query
    query = "SELECT TOP 10* FROM Prices WHERE StockID = " + str(stock_id)
    df = pd.read_sql(query, engine)

    print(df)


All Function List

- db_connection(server_name, database_name) <----------- Connects to Database and Checks Connection
- download_stockData(ticker, start_date = "2020-01-01") <-------------------------------- Downloads Stock Data
- clean_data() <--------------------------------- Cleans Stock Data to Prepare for Ingestion into Database
- insert_sql() <-------------------------- Insert Dataframe into SQL

In [8]:
# ====================================================
# -------- Full Data Pipeline for Ingestion ----------

# Libraries
import pandas as pd
from sqlalchemy import create_engine
import yfinance as yf

# Orchestration Function

def full_ingestion(ticker, stock_id): 
    # Parameters
    start_date = "2020-01-01"
    server_name = r"DESKTOP-EE25GV9\SQLEXPRESS"
    database_name = "stockPredictionApp"

    # Functions
    engine = db_connection(server_name, database_name)# <------------------------- Connects to Database and Checks Connection
    #engine = azure_connection() # <----------------------- For Azure(Cloud) Database
    df = download_stockData(ticker, start_date = start_date)# <----------------------------------------- Downloads Stock Data
    df_cleaned = clean_data(df, stock_id = stock_id)# <------------- Cleans Stock Data to Prepare for Ingestion into Database
    insert_sql(df_cleaned, engine = engine, stock_id = stock_id)# <-------------------------------- Insert Dataframe into SQL


# Call
full_ingestion(ticker = "WFC", stock_id = 3) # <------------------ Change Ticker and Stock ID

Connection successful!

    StockID Ticker       CompanyName      Sector
0        1    BAC   Bank of America  Financials
1        2    TFC  Truist Financial  Financials
2        3    WFC       Wells Fargo  Financials


[*********************100%***********************]  1 of 1 completed


WFC Downloaded! 
 Price           Close       High        Low       Open    Volume
Ticker            WFC        WFC        WFC        WFC       WFC
2020-01-02  45.566654  45.812503  45.363192  45.651428  16803100
2020-01-03  45.286896  45.456447  44.846068  45.024095  15608800
2020-01-06  45.015614  45.100391  44.693470  44.710426  13200300
2020-01-07  44.642597  44.973220  44.481526  44.973220  13278600
2020-01-08  44.778244  45.210599  44.761289  44.795199  16585600
   StockID  PriceDate  OpenPrice  HighPrice   LowPrice  ClosePrice    Volume
0        3 2020-01-02  45.651428  45.812503  45.363192   45.566654  16803100
1        3 2020-01-03  45.024095  45.456447  44.846068   45.286896  15608800
2        3 2020-01-06  44.710426  45.100391  44.693470   45.015614  13200300
3        3 2020-01-07  44.973220  44.973220  44.481526   44.642597  13278600
4        3 2020-01-08  44.795199  45.210599  44.761289   44.778244  16585600
Prices inserted successfully!
   StockID  PriceDate  OpenPrice  H